**Unzip**

In [47]:
import zipfile

with zipfile.ZipFile("ml-100k.zip", 'r') as zip_ref:
    zip_ref.extractall("ml-100k")

**libarires**

In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from sklearn.decomposition import TruncatedSVD

sns.set(style="whitegrid")
plt.rcParams["figure.figsize"] = (7, 5)
RANDOM_STATE = 42


**Load the ratings file**

In [49]:
ratratings = pd.read_csv("ml-100k/ml-100k/u.data", sep="\t",
                      names=["user_id","movie_id","rating","timestamp"])
ratings.head()

,user_id,movie_id,rating,timestamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596


**Load movies file**

In [50]:
movies = pd.read_csv("ml-100k/ml-100k/u.item", sep="|", encoding="latin-1",
                     header=None, usecols=[0,1], names=["movie_id","title"])
movies.head()

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


**Build User–Item Matrix**

In [51]:
user_item_matrix = ratings.pivot(index='user_id', columns='movie_id', values='rating')
user_item_matrix.fillna(0, inplace=True)
user_item_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,4.0,3.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


**Compute User Similarity**

In [52]:
user_similarity = cosine_similarity(user_item_matrix)
user_similarity_df = pd.DataFrame(user_similarity,
                                  index=user_item_matrix.index,
                                  columns=user_item_matrix.index)
user_similarity_df.head()

user_id,1,2,3,4,5,6,7,8,9,10,...,934,935,936,937,938,939,940,941,942,943
user_id,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.166931,0.047460,0.064358,0.378475,0.430239,0.440367,0.319072,0.078138,0.376544,...,0.369527,0.119482,0.274876,0.189705,0.197326,0.118095,0.314072,0.148617,0.179508,0.398175
2,0.166931,1.000000,0.110591,0.178121,0.072979,0.245843,0.107328,0.103344,0.161048,0.159862,...,0.156986,0.307942,0.358789,0.424046,0.319889,0.228583,0.226790,0.161485,0.172268,0.105798
3,0.047460,0.110591,1.000000,0.344151,0.021245,0.072415,0.066137,0.083060,0.061040,0.065151,...,0.031875,0.042753,0.163829,0.069038,0.124245,0.026271,0.161890,0.101243,0.133416,0.026556
4,0.064358,0.178121,0.344151,1.000000,0.031804,0.068044,0.091230,0.188060,0.101284,0.060859,...,0.052107,0.036784,0.133115,0.193471,0.146058,0.030138,0.196858,0.152041,0.170086,0.058752
5,0.378475,0.072979,0.021245,0.031804,1.000000,0.237286,0.373600,0.248930,0.056847,0.201427,...,0.338794,0.080580,0.094924,0.079779,0.148607,0.071459,0.239955,0.139595,0.152497,0.313941


**User-Based Recommendation Function**

In [53]:
def recommend_movies(user_id, n_neighbors=5, n_recommendations=5):
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)[1:n_neighbors+1].index

    neighbor_ratings = user_item_matrix.loc[similar_users].mean(axis=0)

    seen_movies = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    recommendations = neighbor_ratings.drop(seen_movies).sort_values(ascending=False).head(n_recommendations)

    return movies[movies['movie_id'].isin(recommendations.index)]

recommend_movies(user_id=1, n_neighbors=5, n_recommendations=5)

,movie_id,title
272,273,Heat (1995)
432,433,Heathers (1989)
473,474,Dr. Strangelove or: How I Learned to Stop Worr...
565,566,Clear and Present Danger (1994)
731,732,Dave (1993)


**Evaluation: Precision@K**

In [54]:
train, test = train_test_split(ratings, test_size=0.2, random_state=RANDOM_STATE)

train_matrix = train.pivot(index='user_id', columns='movie_id', values='rating').fillna(0)
test_matrix = test.pivot(index='user_id', columns='movie_id', values='rating').fillna(0)

# Recompute similarity on train
user_similarity = cosine_similarity(train_matrix)
user_similarity_df = pd.DataFrame(user_similarity,
                                  index=train_matrix.index,
                                  columns=train_matrix.index)

def precision_at_k(user_id, k=5, n_neighbors=5):
    if user_id not in test_matrix.index:
        return np.nan
    recs = recommend_movies(user_id, n_neighbors=n_neighbors, n_recommendations=k)
    test_movies = test_matrix.loc[user_id][test_matrix.loc[user_id] > 0].index
    hits = len(set(recs['movie_id']).intersection(set(test_movies)))
    return hits / k if k > 0 else 0

# Evaluate on sample users
precisions = [precision_at_k(u, k=5) for u in test_matrix.index[:50]]
print("Average Precision@5:", np.nanmean(precisions))

Average Precision@5: 0.0


**Item-Based CF**

In [55]:
item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(item_similarity,
                                  index=user_item_matrix.columns,
                                  columns=user_item_matrix.columns)

def recommend_similar_movies(movie_id, n=5):
    similar_scores = item_similarity_df[movie_id].sort_values(ascending=False)[1:n+1]
    return movies[movies['movie_id'].isin(similar_scores.index)]

recommend_similar_movies(50, n=5)

,movie_id,title
0,1,Toy Story (1995)
126,127,"Godfather, The (1972)"
171,172,"Empire Strikes Back, The (1980)"
173,174,Raiders of the Lost Ark (1981)
180,181,Return of the Jedi (1983)


**Matrix Factorization (SVD)**

In [56]:
svd = TruncatedSVD(n_components=20, random_state=RANDOM_STATE)
latent_matrix = svd.fit_transform(user_item_matrix)
print("Latent matrix shape:", latent_matrix.shape)

approx_ratings = np.dot(latent_matrix, svd.components_)
approx_ratings_df = pd.DataFrame(approx_ratings,
                                 index=user_item_matrix.index,
                                 columns=user_item_matrix.columns)
approx_ratings_df.head()


Latent matrix shape: (943, 20)


movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,4.228012,2.096937,1.276615,3.139637,0.551456,0.568562,4.419147,2.798920,3.134902,2.193979,...,-0.024103,0.008582,0.014560,0.009707,0.022546,-0.001981,-0.005944,-0.003963,0.031395,0.074819
2,2.024309,-0.008309,0.033837,0.277159,-0.008177,0.341411,1.627962,0.440471,2.541393,0.619027,...,0.000615,-0.021051,-0.008494,-0.005663,-0.000520,0.004875,0.014625,0.009750,-0.004853,-0.028220
3,-0.122395,-0.063842,0.169859,-0.205504,-0.097037,0.016595,-0.293257,-0.073747,-0.435728,0.096075,...,0.004596,-0.010258,0.023943,0.015962,-0.002083,0.011123,0.033370,0.022247,0.003056,0.002247
4,0.449120,-0.178459,0.092678,-0.073230,0.041396,-0.005179,0.338694,-0.103576,-0.099695,-0.169013,...,0.001569,-0.008776,-0.007678,-0.005119,-0.002431,0.005236,0.015708,0.010472,-0.002409,0.000591
5,3.697199,1.322204,0.353221,1.524847,0.507998,-0.143364,2.799689,1.310824,-0.428948,0.384931,...,-0.012855,0.003692,-0.032325,-0.021550,-0.014456,-0.001497,-0.004490,-0.002993,-0.000667,-0.015408


**Recommend using SVD**

In [57]:
def recommend_movies_svd(user_id, n_recommendations=5):
    user_ratings = approx_ratings_df.loc[user_id]
    seen_movies = user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index
    recs = user_ratings.drop(seen_movies).sort_values(ascending=False).head(n_recommendations)
    return movies[movies['movie_id'].isin(recs.index)]

recommend_movies_svd(user_id=1, n_recommendations=5)

,movie_id,title
274,275,Sense and Sensibility (1995)
422,423,E.T. the Extra-Terrestrial (1982)
432,433,Heathers (1989)
474,475,Trainspotting (1996)
482,483,Casablanca (1942)
